# P2 — Mathematical Proof Package

P2는 실험이 아니라, finite graph의 path가 실제 hybrid trajectory가 되고 그 graph 안에서는 Bellman 해가 전역 최적이라는 것을 수학적으로 연결하는 단계다.

In [1]:
from pathlib import Path
import json
from html import escape
import subprocess
import sys
from IPython.display import display, Markdown, Image, HTML, FileLink

def locate_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "p1b_4D").is_dir() and (candidate / "p1b_roadmap_0729.md").exists():
            return candidate
    raise RuntimeError("glider_hybrid_control repository root를 찾지 못했습니다.")

ROOT = locate_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"
STAGE = "P2"
STATUS = "PENDING"

def run_module(module, *arguments, timeout=None):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    completed = subprocess.run(
        command, cwd=ROOT, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"{module} 실행 실패: exit={completed.returncode}")
    return completed.stdout

def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"결과 파일이 없습니다: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def show_png(path, width=1050):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"> ⚠️ 그림이 없습니다: `{path}`"))

def display_table(rows, columns=None):
    if isinstance(rows, dict):
        rows = [rows]
    rows = list(rows)
    if columns is None:
        columns = []
        for row in rows:
            for key in row:
                if key not in columns:
                    columns.append(key)
    if not rows:
        display(Markdown("_(표시할 행이 없습니다.)_"))
        return
    def cell(value):
        if isinstance(value, float):
            value = f"{value:.8g}"
        elif isinstance(value, (dict, list, tuple)):
            value = json.dumps(value, ensure_ascii=False)
        return escape(str(value))
    header = "".join(f"<th>{cell(name)}</th>" for name in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{cell(row.get(name, ''))}</td>" for name in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<div style='overflow-x:auto'><table>"
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    ))

display(Markdown(f"**{STAGE} 상태:** `{STATUS}`  \nRepository: `{ROOT}`"))

**P2 상태:** `PENDING`  
Repository: `C:\Users\jeffe\Desktop\git\glider_hybrid_control`

## 현재 상태

이 단계는 아직 구현 완료되지 않았다. 이 노트북은 완료된 척하지 않고,
구현 목표와 향후 실행 진입점을 한 파일에 고정한다.

**완료 조건**


- graph-path soundness proof
- declared discretization에 대한 relative completeness proof
- finite Bellman/exhaustive-switch exactness proof
- continuous trajectory completeness를 주장하지 않는 명확한 scope

In [2]:
RERUN = False
EXPECTED_MODULE = ROOT / "p1b_4D" / "p2_mathematical_proof_package.py"
RESULT_PATH = ROOT / "results/p2/p2_proof_validation.json"
status = {
    "stage": "P2",
    "implementation module exists": EXPECTED_MODULE.exists(),
    "result exists": RESULT_PATH.exists(),
    "rerun requested": RERUN,
}
display_table(status)

if RERUN:
    if not EXPECTED_MODULE.exists():
        raise RuntimeError(
            f"P2 구현이 아직 없습니다: {EXPECTED_MODULE.name}"
        )
    run_module("p1b_4D.p2_mathematical_proof_package")

if RESULT_PATH.exists():
    payload = load_json(RESULT_PATH)
    display(Markdown("✅ 저장된 결과를 발견했습니다."))
    display(Markdown("```json\n" + json.dumps(payload, ensure_ascii=False, indent=2)[:12000] + "\n```"))
else:
    display(Markdown(
        "> ⏳ **PENDING:** 구현·실험이 완료되면 이 셀이 결과 표와 그림을 바로 표시합니다."
    ))

stage,implementation module exists,result exists,rerun requested
P2,False,False,False


> ⏳ **PENDING:** 구현·실험이 완료되면 이 셀이 결과 표와 그림을 바로 표시합니다.

In [3]:
PROPOSITION = ROOT / "p1b_4D" / "discrete_optimality_proposition.md"
text = PROPOSITION.read_text(encoding="utf-8")
headings = [line for line in text.splitlines() if line.startswith("##")]
display(Markdown("### 현재 proof 초안의 구조\n" + "\n".join(f"- {h}" for h in headings)))
display(FileLink(str(PROPOSITION)))
display(Markdown(
    "> 현재 문서는 finite edge/DAG/Bellman proof를 포함하지만, P2의 graph-level soundness와 relative completeness 패키지는 아직 미완료입니다."
))

### 현재 proof 초안의 구조
- ## 1. Finite follower problem
- ### 1.1 Regular physical-successor actions
- ### 1.2 Edge cost
- ### 1.3 Virtual switching states
- ## 2. Proposition 1: the regular successor graph is a finite DAG
- ## 3. Proposition 2: every admitted edge is execution-consistent
- ## 4. Proposition 3: one backward Bellman sweep is exact on the graph
- ## 5. Proposition 4: exhaustive virtual-switch evaluation gives the finite follower optimum
- ## 6. Exact minimum selection and deterministic tie rule
- ## 7. Complexity
- ## 8. B4 production instance
- ## 9. Scope of the result
- ## 10. Implementation validation

C:\Users\jeffe\Desktop\git\glider_hybrid_control\p1b_4D\discrete_optimality_proposition.md

> 현재 문서는 finite edge/DAG/Bellman proof를 포함하지만, P2의 graph-level soundness와 relative completeness 패키지는 아직 미완료입니다.

## 직관적 목적

P2는 실험이 아니라, finite graph의 path가 실제 hybrid trajectory가 되고 그 graph 안에서는 Bellman 해가 전역 최적이라는 것을 수학적으로 연결하는 단계다.